In [ ]:
import torch
import torch.nn as nn 
from torch.nn import functional as F

block_size = 8
batch_size = 4
max_iters = 100000
learning_rate = 3e-4
n_embd = 384

In [ ]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as file:
    text = file.read()
    chars = sorted(set(text))
    vocab_size = len(chars)

In [ ]:
string_to_int = {char:i for i,char in enumerate(chars)}
int_to_string = {i:char for i,char in enumerate(chars)}

encode = lambda st : [string_to_int[ch] for ch in st]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)

In [ ]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

In [ ]:
@torch.no_grad
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for i in range(eval_iters):
            X, Y = get_batch('split')
            logits, loss = model.forward(X, Y)
            losses[k] = loss.item()
            out[split] = losses.mean()
        model.train()
    

In [ ]:
class GPTLanguageModel(nn.Module):      #creating the Bigram model class
    def __init__(self, vocab_size):
        super().__init__()      #inherit the nn.module parent class using the super keyword
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)  #create 1D n_embd size vectors for each of our 78 chars 
        self.position_embedding_table  = nn.Embedding(block_size, n_embd)    #create 1D n_embd size vectors for each our chars in block_size  (for each letter index)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])    #create four sequential decoder layers using nn.sequential
        self.lm_f = nn.LayerNorm(n_embd) #final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, index, targets = None): 
        logits = self.token_embedding_table(index) #set logits as the tensor
        
        #allocate fixed positional encodings based on sin and cos functions to identify the sequence a particular 
        #sentence follows, how far one word is from another. 
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape          #B = batch_size, T= no of integers or block_size , C = vocab_size
            logits = logits.view(B*T, C)    #adjust the dim of the tensor as per loss function requirement
            targets = targets.view(B*T)     #adjust the target dim as per loss function requirement
            loss = F.cross_entropy(logits, targets) #calculate loss with target values and logits 
            
        return logits, loss 
    
    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :]  #attain the last character's embedded vector for all batches
            probs = F.softmax(logits, dim=-1) #normalise the logits along the last dim or the column dim
            index_next = torch.multinomial(probs, num_samples = 1) #draw one sample of the most probable char
            index = torch.cat((index, index_next), dim = 1) #concatenate that char and repeat
        return index

model = GPTLanguageModel(vocab_size)
    #m = model.to(device)
    
context = torch.zeros((1,1), dtype = torch.long)
generated_chars = decode(model.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)